In [ ]:

# Section 1: Load Training Data and Saved Group-Aware Split
    
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"

DATA = (
    PROJECT_ROOT
    / "1_data_preparation"
    / "data"
    / "processed"
    / "char_tfidf_50k"
)

# Load original training data only
X_train = sparse.load_npz(DATA / "X_train.npz").tocsr()
cost_train = pd.read_pickle(DATA / "train_costs.pkl")
train_questions = pd.read_pickle(DATA / "train_questions.pkl")
preprocessing = joblib.load(DATA / "preprocessing.joblib")

MODELS = list(cost_train.columns)

# Load the exact group-aware split from Gamma regression
split_path = (
    PROJECT_ROOT
    / "saved_models"
    / "training_splits"
    / "routerbench_grouped_subtrain_subval_seed42.npz"
)

with np.load(split_path) as split:
    subtrain_idx = split["subtrain_idx"]
    subval_idx = split["subval_idx"]

# Reconstruct identical partitions
X_subtrain = X_train[subtrain_idx]
X_subval = X_train[subval_idx]

cost_subtrain = cost_train.iloc[subtrain_idx].copy()
cost_subval = cost_train.iloc[subval_idx].copy()

# Verify dimensions and alignment
groups = train_questions["question_group"].to_numpy()

assert X_train.shape == (21900, 50001)
assert cost_train.shape == (21900, 11)
assert X_subtrain.shape == (17520, 50001)
assert X_subval.shape == (4380, 50001)

assert set(groups[subtrain_idx]).isdisjoint(
    set(groups[subval_idx])
)

assert np.isfinite(cost_train.to_numpy()).all()
assert (cost_train.to_numpy() > 0).all()

print("Sub-training features:", X_subtrain.shape)
print("Sub-validation features:", X_subval.shape)
print("Number of LLMs:", len(MODELS))
print("\nGamma training split reproduced successfully.")


Sub-training features: (17520, 50001)
Sub-validation features: (4380, 50001)
Number of LLMs: 11

Gamma training split reproduced successfully.


In [2]:

# Section 2.1: Log-Cost Mean Squared Error

import numpy as np

def log_cost_mse(y_true, predicted_log_cost):
    """
    Mean squared error on the log-cost scale.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    predicted_log_cost = np.asarray(
        predicted_log_cost, dtype=np.float64
    )

    if y_true.shape != predicted_log_cost.shape:
        raise ValueError("Shapes must match.")

    if not np.isfinite(y_true).all():
        raise ValueError("Observed costs must be finite.")

    if (y_true <= 0).any():
        raise ValueError("Observed costs must be positive.")

    if not np.isfinite(predicted_log_cost).all():
        raise ValueError("Predictions must be finite.")

    return float(
        np.mean(
            (np.log(y_true) - predicted_log_cost) ** 2
        )
    )


# Simple correctness test
actual = np.array([1.0, 2.0, 3.0])

perfect_log_predictions = np.log(actual)

imperfect_log_predictions = np.log(
    np.array([1.2, 1.8, 2.5])
)

print(
    "Perfect predictions:",
    log_cost_mse(actual, perfect_log_predictions)
)

print(
    "Imperfect predictions:",
    log_cost_mse(actual, imperfect_log_predictions)
)


Perfect predictions: 0.0
Imperfect predictions: 0.0258610461344085


In [3]:

# Section 2.2: Regularized Log-Normal Objective

import numpy as np

def lognormal_objective(y_true, predicted_log_cost, coefficients, lam):
    """
    Half the mean log-cost MSE plus L2 regularization.
    The intercept is excluded from coefficients.
    """
    if lam < 0:
        raise ValueError("Regularization must be nonnegative.")

    coefficients = np.asarray(coefficients, dtype=np.float64)

    data_loss = 0.5 * log_cost_mse(
        y_true, predicted_log_cost
    )

    penalty = 0.5 * lam * np.dot(
        coefficients, coefficients
    )

    return float(data_loss + penalty)


# Verify on a small example
actual = np.array([1.0, 2.0, 3.0])
predicted_log = np.log(np.array([1.2, 1.8, 2.5]))

beta = np.array([0.5, -0.3, 0.2])
lam = 0.1

print("Log-cost MSE:", log_cost_mse(actual, predicted_log))
print("Half log-cost MSE:",
      0.5 * log_cost_mse(actual, predicted_log))
print("L2 penalty:", 0.5 * lam * np.dot(beta, beta))
print("Regularized objective:",
      lognormal_objective(actual, predicted_log, beta, lam))


Log-cost MSE: 0.0258610461344085
Half log-cost MSE: 0.01293052306720425
L2 penalty: 0.019000000000000003
Regularized objective: 0.03193052306720425


In [4]:

# Section 2.3: Regularized Log-Normal Objective and Gradient

import numpy as np

def lognormal_loss_and_gradient(params, X, y, lam):
    """
    Regularized log-cost regression objective and gradient.

    params[0]  : intercept
    params[1:] : coefficients

    Objective:
        0.5 * mean squared log-cost error
        + 0.5 * lam * ||beta||^2
    """
    if lam < 0:
        raise ValueError("lam must be nonnegative.")

    params = np.asarray(params, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    if np.any(y <= 0) or not np.isfinite(y).all():
        raise ValueError("Costs must be positive and finite.")

    n = X.shape[0]

    intercept = params[0]
    beta = params[1:]

    # Predicted log-cost
    eta = intercept + X @ beta

    # Residuals on the log-cost scale
    residual = eta - np.log(y)

    # Regularized objective
    loss = (
        0.5 * np.mean(residual ** 2)
        + 0.5 * lam * np.dot(beta, beta)
    )

    # Analytical gradient
    grad_intercept = np.mean(residual)

    grad_beta = (
        np.asarray(X.T @ residual).ravel() / n
        + lam * beta
    )

    gradient = np.concatenate([
        [grad_intercept],
        grad_beta
    ])

    return float(loss), gradient


In [5]:

# Section 2.3: Numerical Gradient Verification

from scipy.optimize import check_grad

X_small = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

y_small = np.array([1.0, 2.0, 3.0])

params_small = np.array([0.2, 0.1, -0.1])

lam = 0.1

def objective_only(params):
    return lognormal_loss_and_gradient(
        params, X_small, y_small, lam
    )[0]

def gradient_only(params):
    return lognormal_loss_and_gradient(
        params, X_small, y_small, lam
    )[1]

error = check_grad(
    objective_only,
    gradient_only,
    params_small
)

print("Gradient-check error:", error)


Gradient-check error: 5.556980274123898e-09


In [6]:

# Section 3.1: Test L-BFGS for Log-Normal Regression

import numpy as np
from scipy.optimize import minimize

model_name = "meta/llama-2-70b-chat"

# Use 1,000 sub-training observations
X_fit = X_subtrain[:1000]
y_fit = cost_subtrain[model_name].to_numpy()[:1000]

# Initialize the intercept at the mean log-cost
initial_params = np.zeros(X_fit.shape[1] + 1)
initial_params[0] = np.log(y_fit).mean()

# Fixed regularization strength for this reference model
lam = 0.1

initial_loss, _ = lognormal_loss_and_gradient(
    initial_params, X_fit, y_fit, lam
)

result = minimize(
    fun=lognormal_loss_and_gradient,
    x0=initial_params,
    args=(X_fit, y_fit, lam),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 50,
        "ftol": 1e-9
    }
)

print("Initial objective:", initial_loss)
print("Final objective:", result.fun)
print("Iterations:", result.nit)
print("Converged:", result.success)
print("Message:", result.message)

assert np.isfinite(result.fun)
assert result.fun <= initial_loss + 1e-10


Initial objective: 0.24894710587482013
Final objective: 0.0720898824235029
Iterations: 11
Converged: True
Message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL


In [7]:

# Section 3.2: Log-Normal Regression with Early Stopping

import numpy as np
import pandas as pd
from scipy.optimize import minimize

class LogNormalEarlyStopping(Exception):
    """Signal that early stopping has been triggered."""
    pass


def train_lognormal_early_stopping(
    X_fit,
    y_fit,
    X_holdout,
    y_holdout,
    lam=0.1,
    max_iter=200,
    patience=5,
    min_delta=1e-6
):
    """
    Train L2-regularized log-cost regression.

    Monitor unpenalized log-MSE on sub-training and
    sub-validation data. Return the best checkpoint.
    """

    y_fit = np.asarray(y_fit, dtype=np.float64)
    y_holdout = np.asarray(y_holdout, dtype=np.float64)

    if lam < 0 or patience < 1 or min_delta < 0:
        raise ValueError("Invalid training hyperparameters.")

    if not np.isfinite(y_fit).all() or not np.isfinite(y_holdout).all():
        raise ValueError("Costs must be finite.")

    if (y_fit <= 0).any() or (y_holdout <= 0).any():
        raise ValueError("Costs must be positive.")

    initial_params = np.zeros(X_fit.shape[1] + 1)
    initial_params[0] = np.log(y_fit).mean()

    history = []
    best_params = initial_params.copy()
    best_loss = np.inf
    best_iteration = 0
    no_improvement = 0

    def record_iteration(params):
        nonlocal best_params, best_loss
        nonlocal best_iteration, no_improvement

        iteration = len(history)

        eta_fit = params[0] + X_fit @ params[1:]
        eta_holdout = params[0] + X_holdout @ params[1:]

        train_loss = log_cost_mse(y_fit, eta_fit)
        val_loss = log_cost_mse(y_holdout, eta_holdout)

        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise FloatingPointError("Non-finite loss.")

        history.append({
            "iteration": iteration,
            "subtrain_log_mse": train_loss,
            "subval_log_mse": val_loss
        })

        # Preserve the lowest observed sub-validation loss
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = params.copy()
            best_iteration = iteration

        # Patience counts iterations without meaningful improvement
        if iteration == 0:
            record_iteration.reference_loss = val_loss
        elif val_loss < record_iteration.reference_loss - min_delta:
            record_iteration.reference_loss = val_loss
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            raise LogNormalEarlyStopping()

    # Record initial state
    record_iteration(initial_params)

    stopped_early = False
    optimizer_result = None

    try:
        optimizer_result = minimize(
            fun=lognormal_loss_and_gradient,
            x0=initial_params,
            args=(X_fit, y_fit, lam),
            method="L-BFGS-B",
            jac=True,
            callback=record_iteration,
            options={
                "maxiter": max_iter,
                "ftol": 1e-12,
                "gtol": 1e-8
            }
        )

    except LogNormalEarlyStopping:
        stopped_early = True

    return {
        "params": best_params,
        "best_iteration": best_iteration,
        "best_subval_log_mse": best_loss,
        "history": pd.DataFrame(history),
        "stopped_early": stopped_early,
        "optimizer_result": optimizer_result
    }


In [8]:

# Section 3.2: Test Log-Normal Early Stopping

model_name = "meta/llama-2-70b-chat"

lognormal_test_run = train_lognormal_early_stopping(
    X_fit=X_subtrain,
    y_fit=cost_subtrain[model_name].to_numpy(),
    X_holdout=X_subval,
    y_holdout=cost_subval[model_name].to_numpy(),
    lam=0.1,
    max_iter=200,
    patience=5,
    min_delta=1e-6
)

print("Best iteration:",
      lognormal_test_run["best_iteration"])

print("Best sub-validation log-MSE:",
      lognormal_test_run["best_subval_log_mse"])

print("Stopped early:",
      lognormal_test_run["stopped_early"])

print("Iterations recorded:",
      len(lognormal_test_run["history"]) - 1)

print("\nFinal recorded losses:")
display(lognormal_test_run["history"].tail())


Best iteration: 5
Best sub-validation log-MSE: 0.09100901408105601
Stopped early: True
Iterations recorded: 10

Final recorded losses:


,iteration,subtrain_log_mse,subval_log_mse
6,6,0.087291,0.092877
7,7,0.087200,0.092774
8,8,0.087050,0.092612
9,9,0.087049,0.092612
10,10,0.087049,0.092612


In [10]:

# Section 4.1: Train Log-Normal Regression Models for All 11 LLMs

import numpy as np
import pandas as pd

LOGNORMAL_LAMBDA = 0.1

lognormal_final_models = {}
lognormal_training_histories = {}
lognormal_training_records = []

for model_name in MODELS:

    print(f"Training: {model_name}")

    result = train_lognormal_early_stopping(
        X_fit=X_subtrain,
        y_fit=cost_subtrain[model_name].to_numpy(),
        X_holdout=X_subval,
        y_holdout=cost_subval[model_name].to_numpy(),
        lam=LOGNORMAL_LAMBDA,
        max_iter=200,
        patience=5,
        min_delta=1e-6
    )

    # Preserve the best checkpoint
    lognormal_final_models[model_name] = result["params"].copy()

    # Preserve the full loss history
    lognormal_training_histories[model_name] = result["history"].copy()

    lognormal_training_records.append({
        "model": model_name,
        "lambda": LOGNORMAL_LAMBDA,
        "best_iteration": result["best_iteration"],
        "best_subval_log_mse": result["best_subval_log_mse"],
        "iterations_recorded": len(result["history"]) - 1,
        "stopped_early": result["stopped_early"]
    })

lognormal_training_summary = pd.DataFrame(
    lognormal_training_records
)

print("\nAll 11 log-normal regressions trained.")
display(lognormal_training_summary)


Training: WizardLM/WizardLM-13B-V1.2
Training: claude-instant-v1
Training: claude-v1
Training: claude-v2
Training: gpt-3.5-turbo-1106
Training: gpt-4-1106-preview
Training: meta/code-llama-instruct-34b-chat
Training: meta/llama-2-70b-chat
Training: mistralai/mistral-7b-chat
Training: mistralai/mixtral-8x7b-chat
Training: zero-one-ai/Yi-34B-Chat

All 11 log-normal regressions trained.


,model,lambda,best_iteration,best_subval_log_mse,iterations_recorded,stopped_early
0,WizardLM/WizardLM-13B-V1.2,0.1,5,0.097665,10,True
1,claude-instant-v1,0.1,8,0.168308,13,True
2,claude-v1,0.1,5,0.151859,10,True
3,claude-v2,0.1,8,0.195077,13,True
4,gpt-3.5-turbo-1106,0.1,5,0.104338,10,True
5,gpt-4-1106-preview,0.1,8,0.237798,13,True
6,meta/code-llama-instruct-34b-chat,0.1,5,0.070043,10,True
7,meta/llama-2-70b-chat,0.1,5,0.091009,10,True
8,mistralai/mistral-7b-chat,0.1,5,0.080629,10,True
9,mistralai/mixtral-8x7b-chat,0.1,5,0.084141,10,True


In [11]:

# Section 4.2: Verify All Log-Normal Checkpoints

import numpy as np
import pandas as pd

verification_records = []

for model_name in MODELS:

    params = lognormal_final_models[model_name]
    history = lognormal_training_histories[model_name]

    # Verify parameter dimensions and numerical validity
    assert params.shape == (X_subtrain.shape[1] + 1,)
    assert np.isfinite(params).all()

    # Predict log-cost using the saved best checkpoint
    predicted_log_cost = (
        params[0] + X_subval @ params[1:]
    )

    assert np.isfinite(predicted_log_cost).all()

    # Recalculate sub-validation log-MSE
    y_subval = cost_subval[model_name].to_numpy()

    verified_loss = log_cost_mse(
        y_subval,
        predicted_log_cost
    )

    # Compare against recorded minimum
    best_recorded_loss = history["subval_log_mse"].min()

    np.testing.assert_allclose(
        verified_loss,
        best_recorded_loss,
        rtol=1e-8,
        atol=1e-10
    )

    verification_records.append({
        "model": model_name,
        "parameter_count": len(params),
        "best_subval_log_mse": verified_loss,
        "verified": True
    })

lognormal_verification = pd.DataFrame(
    verification_records
)

display(lognormal_verification)

print("\nAll 11 log-normal checkpoints verified successfully.")


,model,parameter_count,best_subval_log_mse,verified
0,WizardLM/WizardLM-13B-V1.2,50002,0.097665,True
1,claude-instant-v1,50002,0.168308,True
2,claude-v1,50002,0.151859,True
3,claude-v2,50002,0.195077,True
4,gpt-3.5-turbo-1106,50002,0.104338,True
5,gpt-4-1106-preview,50002,0.237798,True
6,meta/code-llama-instruct-34b-chat,50002,0.070043,True
7,meta/llama-2-70b-chat,50002,0.091009,True
8,mistralai/mistral-7b-chat,50002,0.080629,True
9,mistralai/mixtral-8x7b-chat,50002,0.084141,True



All 11 log-normal checkpoints verified successfully.


In [12]:

# Section 5.1: Export Final Early-Stopped Log-Normal Models

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

EXPORT_DIR = PROJECT_ROOT / "saved_models" / "lognormal_final"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Combine training histories for all 11 models
all_histories = pd.concat(
    [
        history.assign(model=model_name)
        for model_name, history in lognormal_training_histories.items()
    ],
    ignore_index=True
)

# Package models, preprocessing, and training metadata
lognormal_final_bundle = {
    "model_type": "lognormal_log_cost_l2",
    "feature_type": "char_tfidf_50k_plus_length",
    "model_names": list(MODELS),
    "models": lognormal_final_models,
    "preprocessing": preprocessing,
    "lambda": LOGNORMAL_LAMBDA,
    "training_summary": lognormal_training_summary,
    "feature_count": X_subtrain.shape[1],
    "early_stopping": {
        "patience": 5,
        "min_delta": 1e-6,
        "max_iter": 200,
        "monitor": "subvalidation_log_mse"
    },
    "training_split_file": (
        "saved_models/training_splits/"
        "routerbench_grouped_subtrain_subval_seed42.npz"
    ),
    "prediction_target": "log_inference_cost",
    "prediction_link": "identity_on_log_scale",
    "calibration_status": "not_calibrated"
}

# Save model bundle
model_path = (
    EXPORT_DIR / "lognormal_tfidf_length_early_stopped.joblib"
)

joblib.dump(
    lognormal_final_bundle,
    model_path,
    compress=3
)

# Save training histories
history_path = (
    EXPORT_DIR / "lognormal_tfidf_length_training_history.csv"
)

all_histories.to_csv(history_path, index=False)

# Save training summary
summary_path = (
    EXPORT_DIR / "lognormal_tfidf_length_training_summary.csv"
)

lognormal_training_summary.to_csv(summary_path, index=False)

print("Log-normal training artifacts saved successfully!")

for path in [model_path, history_path, summary_path]:
    print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MiB")
    print("Location:", path)


Log-normal training artifacts saved successfully!
lognormal_tfidf_length_early_stopped.joblib — 4.39 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/lognormal_final/lognormal_tfidf_length_early_stopped.joblib
lognormal_tfidf_length_training_history.csv — 0.01 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/lognormal_final/lognormal_tfidf_length_training_history.csv
lognormal_tfidf_length_training_summary.csv — 0.00 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/lognormal_final/lognormal_tfidf_length_training_summary.csv


In [13]:

# Section 5.2: Reload and Verify Saved Log-Normal Models

from pathlib import Path
import joblib
import numpy as np

model_path = (
    PROJECT_ROOT
    / "saved_models"
    / "lognormal_final"
    / "lognormal_tfidf_length_early_stopped.joblib"
)

# Reload the model bundle
loaded_bundle = joblib.load(model_path)

# Verify model count and feature dimensions
assert len(loaded_bundle["models"]) == 11
assert loaded_bundle["feature_count"] == 50001
assert loaded_bundle["calibration_status"] == "not_calibrated"

# Use a small sample from the sub-training partition
X_sample = X_subtrain[:5]

for model_name in MODELS:

    original_params = lognormal_final_models[model_name]
    loaded_params = loaded_bundle["models"][model_name]

    # Verify the saved coefficients match exactly
    np.testing.assert_array_equal(
        original_params,
        loaded_params
    )

    # Predict log-costs using original coefficients
    original_predictions = (
        original_params[0]
        + X_sample @ original_params[1:]
    )

    # Predict log-costs using reloaded coefficients
    loaded_predictions = (
        loaded_params[0]
        + X_sample @ loaded_params[1:]
    )

    # Verify identical predictions
    np.testing.assert_allclose(
        original_predictions,
        loaded_predictions,
        rtol=1e-12,
        atol=1e-15
    )

print("Log-normal model bundle loaded successfully.")
print("All 11 parameter vectors verified.")
print("Reloaded predictions match original predictions.")
print("Calibration status: Not calibrated.")


Log-normal model bundle loaded successfully.
All 11 parameter vectors verified.
Reloaded predictions match original predictions.
Calibration status: Not calibrated.
